In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_scatter import scatter
from torch_geometric.utils import softmax as pyg_softmax
import timeit
import os
import warnings
warnings.filterwarnings('ignore')

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

print("\n" + "="*80)
print("QUICK TEST MODE: Single Epoch on Small Batch")
print("="*80)

Using device: cpu

QUICK TEST MODE: Single Epoch on Small Batch


In [23]:
print("\n[STEP 1/5] Loading data...")
try:
    train_loader, test_loader, train_dataset, test_dataset = load_real_dataset(
        dataset='Davis',
        fold=1,
        batch_size=2,  # Small batch
        num_workers=0,
        compound_node_dim=44,
        compound_edge_dim=10,
        protein_node_dim=1321,
        protein_edge_dim=5
    )
    print(f"✓ Data loaded successfully")
    print(f"  - Train batches: {len(train_loader)}")
    print(f"  - Test batches: {len(test_loader)}")
except Exception as e:
    print(f"✗ Error loading data: {e}")
    import traceback
    traceback.print_exc()


[STEP 1/5] Loading data...
Loading CSV from: AttentionMGT/Data/Davis/DTA/fold/1/Davis_train.csv
✓ Loaded train split: 19964 samples
  - Data directory: AttentionMGT/
  - Compound graphs dir: AttentionMGT/data/Davis/Processed/compound_graph/
  - Compound graphs available: True
  - Protein graphs available: True
  - ESM embeddings available: True
Loading CSV from: AttentionMGT/Data/Davis/DTA/fold/1/Davis_test.csv
✓ Loaded test split: 4992 samples
  - Data directory: AttentionMGT/
  - Compound graphs dir: AttentionMGT/data/Davis/Processed/compound_graph/
  - Compound graphs available: True
  - Protein graphs available: True
  - ESM embeddings available: True
✓ Data loaded successfully
  - Train batches: 9982
  - Test batches: 2496


In [24]:
# Check what dimension your protein graphs actually have
protein_id = train_dataset.protein_ids[2]
protein_path = f'AttentionMGT/Data/Davis/processed/pocket_graph/{protein_id}.pt'

if os.path.exists(protein_path):
    pg = torch.load(protein_path, weights_only=False)
    print(f"✓ Protein graph loaded!")
    print(f"  Node features shape: {pg.x.shape}")
    print(f"  Actual feature dimension: {pg.x.shape[1]}")
    print(f"  This is what you should use for node_dim!")
else:
    print(f"✗ File not found: {protein_path}")

✓ Protein graph loaded!
  Node features shape: torch.Size([18, 1321])
  Actual feature dimension: 1321
  This is what you should use for node_dim!


In [25]:
print("\n[STEP 2/5] Initializing model...")
try:
    model = MDGTDTInet(
        compound_dim=128,
        protein_dim=128,
        gt_layers=10,
        gt_heads=8,
        out_dim=1,
        compound_node_dim=44,
        compound_edge_dim=10,
        protein_node_dim=1321,
        protein_edge_dim=5,
        device=device
    )
    model.to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"✓ Model initialized")
    print(f"  - Parameters: {total_params:,}")
    print(f"  - Device: {device}")
except Exception as e:
    print(f"✗ Error initializing model: {e}")
    import traceback
    traceback.print_exc()


[STEP 2/5] Initializing model...
✓ Model initialized
  - Parameters: 6,392,449
  - Device: cpu


In [26]:
print("\n[STEP 3/5] Setting up optimizer...")
try:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    print(f"✓ Optimizer ready (Adam, lr=1e-4)")
except Exception as e:
    print(f"✗ Error: {e}")


[STEP 3/5] Setting up optimizer...
✓ Optimizer ready (Adam, lr=1e-4)


In [27]:
print("\n[STEP 4/5] Running 1 training epoch...")
try:
    epoch_start = timeit.default_timer()
    train_loss = train(model, device, train_loader, optimizer, criterion)
    epoch_time = timeit.default_timer() - epoch_start
    print(f"✓ Training epoch completed")
    print(f"  - Time: {epoch_time:.2f} seconds")
    print(f"  - Loss: {train_loss:.6f}")
except Exception as e:
    print(f"✗ Error during training: {e}")
    import traceback
    traceback.print_exc()


[STEP 4/5] Running 1 training epoch...
  Batch 10/9982, Loss: 31.165939
  Batch 20/9982, Loss: 13.723335
  Batch 30/9982, Loss: 1.050946
  Batch 40/9982, Loss: 6.753105
  Batch 50/9982, Loss: 2.717457
  Batch 60/9982, Loss: 1.042715
  Batch 70/9982, Loss: 5.678865
  Batch 80/9982, Loss: 2.858230
  Batch 90/9982, Loss: 0.146655
  Batch 100/9982, Loss: 0.677709
  Batch 110/9982, Loss: 3.145928
  Batch 120/9982, Loss: 0.395190
  Batch 130/9982, Loss: 2.280035
  Batch 140/9982, Loss: 0.563622
  Batch 150/9982, Loss: 0.299177
  Batch 160/9982, Loss: 6.353270
  Batch 170/9982, Loss: 0.062043
  Batch 180/9982, Loss: 0.293264
  Batch 190/9982, Loss: 0.603196
  Batch 200/9982, Loss: 0.008058
  Batch 210/9982, Loss: 1.523082
  Batch 220/9982, Loss: 1.260069
  Batch 230/9982, Loss: 0.152470
  Batch 240/9982, Loss: 2.058742
  Batch 250/9982, Loss: 2.684692
  Batch 260/9982, Loss: 0.422010
  Batch 270/9982, Loss: 1.850911
  Batch 280/9982, Loss: 0.061052
  Batch 290/9982, Loss: 2.650680
  Batch 30

In [28]:
print("\n[STEP 5/5] Running 1 test epoch...")
try:
    test_start = timeit.default_timer()
    mse_test, rmse_test, ci_test, rm2_test = test(model, device, test_loader)
    test_time = timeit.default_timer() - test_start
    print(f"✓ Test epoch completed")
    print(f"  - Time: {test_time:.2f} seconds")
    print(f"  - MSE: {mse_test:.6f}")
    print(f"  - RMSE: {rmse_test:.6f}")
    print(f"  - CI: {ci_test:.5f}")
    print(f"  - RM²: {rm2_test:.5f}")
except Exception as e:
    print(f"✗ Error during testing: {e}")
    import traceback
    traceback.print_exc()


[STEP 5/5] Running 1 test epoch...
  🔍 Starting test phase with 2496 batches...
  📊 Batch 0 shapes:
    - protein_embedding shape: torch.Size([2, 239, 320])
    - label shape: torch.Size([2])
    - compound_graph.num_nodes: 73
    - protein_graph.num_nodes: 257
    - output shape: torch.Size([2, 1])
    - output values (first 3): tensor([[5.6632],
        [5.6735]])
  ✓ Processed 50/2496 test batches
  ✓ Processed 100/2496 test batches
  ✓ Processed 150/2496 test batches
  ✓ Processed 200/2496 test batches
  ✓ Processed 250/2496 test batches
  ✓ Processed 300/2496 test batches
  ✓ Processed 350/2496 test batches
  ✓ Processed 400/2496 test batches
  ✓ Processed 450/2496 test batches
  ✓ Processed 500/2496 test batches
  ✓ Processed 550/2496 test batches
  ✓ Processed 600/2496 test batches
  ✓ Processed 650/2496 test batches
  ✓ Processed 700/2496 test batches
  ✓ Processed 750/2496 test batches
  ✓ Processed 800/2496 test batches
  ✓ Processed 850/2496 test batches
  ✓ Processed 900/2

In [29]:
print("\n" + "="*80)
print("✅ QUICK TEST COMPLETE!")
print("="*80)
print("\nIf all 5 steps passed, your pipeline is working!")
print("\nNext steps:")
print("  1. Go to Section 1 below for full training")
print("  2. Set epochs=50 for full training")
print(f"  3. Estimated time: ~{epoch_time * 50 / 60:.1f} minutes on {device}")
print("\n" + "="*80)


✅ QUICK TEST COMPLETE!

If all 5 steps passed, your pipeline is working!

Next steps:
  1. Go to Section 1 below for full training
  2. Set epochs=50 for full training
  3. Estimated time: ~3788.6 minutes on cpu



## Section 1: Real Dataset Loader

Load preprocessed data from AttentionMGT notebook

In [6]:
class RealDTIDatasetPyG(Dataset):
    """Load real preprocessed data from AttentionMGT notebook output"""
    def __init__(self, dataset='Davis', fold=1, split='train', data_dir='AttentionMGT/'):
        """
        Args:
            dataset: 'Davis' or 'KIBA'
            fold: fold number (1-5)
            split: 'train' or 'test'
            data_dir: path to data directory
        """
        self.dataset = dataset
        self.fold = fold
        self.split = split
        
        # Base paths - compound graphs are in data/Davis/Processed, others in Data/Davis/processed
        self.compound_graph_dir = f'{data_dir}data/{dataset}/Processed/compound_graph/'
        self.processed_dir = f'{data_dir}Data/{dataset}/processed/'
        self.protein_graph_dir = f'{self.processed_dir}pocket_graph/'
        self.esm_dir = f'{self.processed_dir}ESM_embedding_pocket/'
        self.fold_dir = f'{data_dir}Data/{dataset}/DTA/fold/{fold}/'
        
        # Load metadata from fold directory
        self.csv_file = f'{self.fold_dir}{dataset}_{split}.csv'
        print(f"Loading CSV from: {self.csv_file}")
        self.df = pd.read_csv(self.csv_file)
        
        # Load compound IDs and protein IDs
        self.compound_ids = self.df['COMPOUND_ID'].values
        self.protein_ids = self.df['PROTEIN_ID'].values
        self.labels = self.df['REG_LABEL'].values
        
        # Check what files are available
        self.has_compound_graphs = os.path.exists(self.compound_graph_dir)
        self.has_protein_graphs = os.path.exists(self.protein_graph_dir)
        self.has_esm = os.path.exists(self.esm_dir)
        
        print(f"✓ Loaded {split} split: {len(self.df)} samples")
        print(f"  - Data directory: {data_dir}")
        print(f"  - Compound graphs dir: {self.compound_graph_dir}")
        print(f"  - Compound graphs available: {self.has_compound_graphs}")
        print(f"  - Protein graphs available: {self.has_protein_graphs}")
        print(f"  - ESM embeddings available: {self.has_esm}")
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        compound_id = self.compound_ids[idx]
        protein_id = self.protein_ids[idx]
        label = self.labels[idx]
        
        # Load compound graph
        if self.has_compound_graphs:
            compound_graph_path = f'{self.compound_graph_dir}{compound_id}.pt'
            if os.path.exists(compound_graph_path):
                compound_graph = torch.load(compound_graph_path, weights_only=False)
                # Fix attribute name: laplacian_pe -> lap_pos_enc
                if hasattr(compound_graph, 'laplacian_pe') and not hasattr(compound_graph, 'lap_pos_enc'):
                    compound_graph.lap_pos_enc = compound_graph.laplacian_pe
            else:
                compound_graph = self._create_dummy_compound_graph()
        else:
            compound_graph = self._create_dummy_compound_graph()
        
        # Load protein graph
        if self.has_protein_graphs:
            protein_graph_path = f'{self.protein_graph_dir}{protein_id}.pt'
            if os.path.exists(protein_graph_path):
                protein_graph = torch.load(protein_graph_path, weights_only=False)
            else:
                protein_graph = self._create_dummy_protein_graph()
        else:
            protein_graph = self._create_dummy_protein_graph()
        
        # Load ESM embedding
        if self.has_esm:
            esm_path = f'{self.esm_dir}{protein_id}.npy'
            if os.path.exists(esm_path):
                protein_embedding = np.load(esm_path, allow_pickle=True).astype(np.float32)
            else:
                protein_embedding = np.random.randn(150, 320).astype(np.float32)
        else:
            protein_embedding = np.random.randn(150, 320).astype(np.float32)
        
        return compound_graph, protein_graph, protein_embedding, label
    
    def _create_dummy_compound_graph(self):
        """Fallback compound graph with typical dimensions"""
        x = torch.randn(50, 44)  # ~50 atoms with 44 features
        edge_index = torch.randint(0, 50, (2, 150))
        edge_attr = torch.randn(150, 10)
        lap_pos_enc = torch.randn(50, 8)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, lap_pos_enc=lap_pos_enc)
    
    def _create_dummy_protein_graph(self):
        """Fallback protein graph with CORRECT 1321-D dimensions (41 structural + 1280 ESM)"""
        # Use 1321-D to match real pocket graphs: 41 (structural) + 1280 (ESM embeddings)
        x = torch.randn(150, 1321)  # 150 residues with 1321 features (NOT 41!)
        edge_index = torch.randint(0, 150, (2, 300))
        edge_attr = torch.randn(300, 5)
        lap_pos_enc = torch.randn(150, 8)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, lap_pos_enc=lap_pos_enc)


def custom_collate_pyg(batch):  
    compound_graphs = [item[0] for item in batch]
    protein_graphs = [item[1] for item in batch]
    protein_embeddings = [item[2] for item in batch]
    labels = [item[3] for item in batch]

    compound_batch = Batch.from_data_list(compound_graphs)
    protein_batch = Batch.from_data_list(protein_graphs)

    protein_lens = [g.num_nodes for g in protein_graphs]
    max_protein_len = max(protein_lens)

    padded_embeddings = []
    for emb in protein_embeddings:
        if emb.shape[0] < max_protein_len:
            pad_size = max_protein_len - emb.shape[0]
            padded = np.pad(emb, ((0, pad_size), (0, 0)), mode='constant', constant_values=0)
            padded_embeddings.append(padded)
        else:
            padded_embeddings.append(emb[:max_protein_len])  # Truncate if too long

    protein_embedding_batch = torch.FloatTensor(np.array(padded_embeddings))
    label_batch = torch.FloatTensor(labels)
    return compound_batch, protein_batch, protein_embedding_batch, label_batch


def load_real_dataset(dataset='Davis', fold=1, batch_size=4, num_workers=0, compound_node_dim=44, compound_edge_dim=10, protein_node_dim=41, protein_edge_dim=5):
    """
    Load real preprocessed dataset from AttentionMGT notebook
    
    Args:
        dataset: 'Davis' or 'KIBA'
        fold: fold number (1-5)
        batch_size: batch size for DataLoader
        num_workers: number of workers for DataLoader
        compound_node_dim: node feature dimension for compound graphs
        compound_edge_dim: edge feature dimension for compound graphs
        protein_node_dim: node feature dimension for protein graphs
        protein_edge_dim: edge feature dimension for protein graphs
    
    Returns:
        train_loader, test_loader, train_dataset, test_dataset
    """
    # Load datasets
    train_dataset = RealDTIDatasetPyG(dataset=dataset, fold=fold, split='train')
    test_dataset = RealDTIDatasetPyG(dataset=dataset, fold=fold, split='test')
    
    # Create DataLoaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        collate_fn=custom_collate_pyg, 
        drop_last=True,
        num_workers=num_workers
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        collate_fn=custom_collate_pyg, 
        drop_last=False,
        num_workers=num_workers
    )
    
    return train_loader, test_loader, train_dataset, test_dataset

## Section 2: Model Architecture

Load the MDGTDTInet model

In [7]:
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, num_heads, use_bias):
        super().__init__()
        
        self.out_dim = out_dim
        self.num_heads = num_heads
        
        if use_bias:
            self.Q = nn.Linear(in_dim, out_dim * num_heads, bias=True)
            self.K = nn.Linear(in_dim, out_dim * num_heads, bias=True)
            self.V = nn.Linear(in_dim, out_dim * num_heads, bias=True)
            self.proj_e = nn.Linear(in_dim, out_dim * num_heads, bias=True)
        else:
            self.Q = nn.Linear(in_dim, out_dim * num_heads, bias=False)
            self.K = nn.Linear(in_dim, out_dim * num_heads, bias=False)
            self.V = nn.Linear(in_dim, out_dim * num_heads, bias=False)
            self.proj_e = nn.Linear(in_dim, out_dim * num_heads, bias=False)
    
    def forward(self, h, edge_index, e):
        num_nodes = h.size(0)
        num_edges = edge_index.size(1)
        
        # Project to Q, K, V
        Q_h = self.Q(h).view(num_nodes, self.num_heads, self.out_dim)
        K_h = self.K(h).view(num_nodes, self.num_heads, self.out_dim)
        V_h = self.V(h).view(num_nodes, self.num_heads, self.out_dim)
        proj_e = self.proj_e(e).view(num_edges, self.num_heads, self.out_dim)
        
        # Get source and target node indices
        src, dst = edge_index[0], edge_index[1]
        
        # Gather K, Q, V for each edge
        K_j = K_h[src]  # [num_edges, num_heads, out_dim]
        Q_i = Q_h[dst]  # [num_edges, num_heads, out_dim]
        V_j = V_h[src]  # [num_edges, num_heads, out_dim]
        
        # Compute attention scores: Q·K / sqrt(d_k)
        score = (Q_i * K_j) / (self.out_dim ** 0.5)  # [num_edges, num_heads, out_dim]
        
        # Modulate by edge features (element-wise multiplication)
        score = score * proj_e  # [num_edges, num_heads, out_dim]
        
        # Store edge output (before softmax)
        e_out = score
        
        # Sum across feature dimension for softmax
        score_scalar = score.sum(dim=-1)  # [num_edges, num_heads]
        
        # Clamp for numerical stability
        score_scalar = score_scalar.clamp(-5, 5)
        
        # Apply softmax per target node for each head separately
        attn_weights = torch.zeros_like(score_scalar)
        for head in range(self.num_heads):
            attn_weights[:, head] = pyg_softmax(score_scalar[:, head], dst, num_nodes=num_nodes)
        
        # Expand for multiplication with V
        attn = attn_weights.unsqueeze(-1)  # [num_edges, num_heads, 1]
        
        # Weight the values
        msg = V_j * attn  # [num_edges, num_heads, out_dim]
        
        # Aggregate messages to each node
        h_out = scatter(msg, dst, dim=0, dim_size=num_nodes, reduce='sum')
        
        # Compute normalization factor (sum of attention weights)
        z = scatter(attn, dst, dim=0, dim_size=num_nodes, reduce='sum')
        
        # Normalize
        h_out = h_out / (z + 1e-6)
        
        return h_out, e_out

In [8]:
class GraphTransformerLayer(nn.Module):
    def __init__(self, in_dim, out_dim, num_heads, dropout, layer_norm=True, batch_norm=False, residual=True, use_bias=False):
        super().__init__()
        self.in_channels = in_dim
        self.out_channels = out_dim
        self.num_heads = num_heads
        self.dropout = dropout
        self.residual = residual
        self.layer_norm = layer_norm
        self.batch_norm = batch_norm
        
        self.attention = MultiHeadAttentionLayer(in_dim, out_dim // num_heads, num_heads, use_bias)
        
        self.O_h = nn.Linear(out_dim, out_dim)
        self.O_e = nn.Linear(out_dim, out_dim)
        
        if self.layer_norm:
            self.layer_norm1_h = nn.LayerNorm(out_dim)
            self.layer_norm1_e = nn.LayerNorm(out_dim)
        
        if self.batch_norm:
            self.batch_norm1_h = nn.BatchNorm1d(out_dim)
            self.batch_norm1_e = nn.BatchNorm1d(out_dim)
        
        self.FFN_h_layer1 = nn.Linear(out_dim, out_dim * 2)
        self.FFN_h_layer2 = nn.Linear(out_dim * 2, out_dim)
        
        self.FFN_e_layer1 = nn.Linear(out_dim, out_dim * 2)
        self.FFN_e_layer2 = nn.Linear(out_dim * 2, out_dim)
        
        if self.layer_norm:
            self.layer_norm2_h = nn.LayerNorm(out_dim)
            self.layer_norm2_e = nn.LayerNorm(out_dim)
        
        if self.batch_norm:
            self.batch_norm2_h = nn.BatchNorm1d(out_dim)
            self.batch_norm2_e = nn.BatchNorm1d(out_dim)
    
    def forward(self, h, edge_index, e):
        h_in1 = h
        e_in1 = e
        
        # Multi-head attention
        h_attn_out, e_attn_out = self.attention(h, edge_index, e)
        
        h = h_attn_out.view(-1, self.out_channels)
        e = e_attn_out.view(-1, self.out_channels)
        
        h = F.dropout(h, self.dropout, training=self.training)
        e = F.dropout(e, self.dropout, training=self.training)
        
        h = self.O_h(h)
        e = self.O_e(e)
        
        if self.residual:
            h = h_in1 + h
            e = e_in1 + e
        
        if self.layer_norm:
            h = self.layer_norm1_h(h)
            e = self.layer_norm1_e(e)
        
        if self.batch_norm:
            h = self.batch_norm1_h(h)
            e = self.batch_norm1_e(e)
        
        h_in2 = h
        e_in2 = e
        
        # FFN for h
        h = self.FFN_h_layer1(h)
        h = F.relu(h)
        h = F.dropout(h, self.dropout, training=self.training)
        h = self.FFN_h_layer2(h)
        
        # FFN for e
        e = self.FFN_e_layer1(e)
        e = F.relu(e)
        e = F.dropout(e, self.dropout, training=self.training)
        e = self.FFN_e_layer2(e)
        
        if self.residual:
            h = h_in2 + h
            e = e_in2 + e
        
        if self.layer_norm:
            h = self.layer_norm2_h(h)
            e = self.layer_norm2_e(e)
        
        if self.batch_norm:
            h = self.batch_norm2_h(h)
            e = self.batch_norm2_e(e)
        
        return h, e

In [9]:
class GraphTransformerCompound(nn.Module):
    def __init__(self, device, n_layers, node_dim, edge_dim, hidden_dim, out_dim, n_heads, in_feat_dropout, dropout, pos_enc_dim):
        super().__init__()
        self.device = device
        self.layer_norm = True
        self.batch_norm = False
        self.residual = True
        self.linear_h = nn.Linear(node_dim, hidden_dim)
        self.linear_e = nn.Linear(edge_dim, hidden_dim)
        self.in_feat_dropout = nn.Dropout(in_feat_dropout)
        self.embedding_lap_pos_enc = nn.Linear(pos_enc_dim, hidden_dim)
        
        self.layers = nn.ModuleList([
            GraphTransformerLayer(hidden_dim, hidden_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
            for _ in range(n_layers - 1)
        ])
        self.layers.append(
            GraphTransformerLayer(hidden_dim, out_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
        )
    
    def forward(self, data):
        data = data.to(self.device)
        h = data.x.float().to(self.device)
        h_lap_pos_enc = data.lap_pos_enc.to(self.device)
        e = data.edge_attr.float().to(self.device)
        edge_index = data.edge_index.to(self.device)
        
        h = self.linear_h(h)
        h_lap_pos_enc = self.embedding_lap_pos_enc(h_lap_pos_enc.float())
        h = h + h_lap_pos_enc
        h = self.in_feat_dropout(h)
        
        e = self.linear_e(e)
        
        for conv in self.layers:
            h, e = conv(h, edge_index, e)
        
        return h

In [10]:
class GraphTransformerProtein(nn.Module):
    def __init__(self, device, n_layers, node_dim, edge_dim, hidden_dim, out_dim, n_heads, in_feat_dropout, dropout, pos_enc_dim):
        super().__init__()
        self.device = device
        self.layer_norm = True
        self.batch_norm = False
        self.residual = True
        self.linear_h = nn.Linear(node_dim, hidden_dim)  # ✅ Use node_dim parameter (1321D)
        self.linear_e = nn.Linear(edge_dim, hidden_dim)
        self.in_feat_dropout = nn.Dropout(in_feat_dropout)
        self.embedding_lap_pos_enc = nn.Linear(pos_enc_dim, hidden_dim)
        
        self.layers = nn.ModuleList([
            GraphTransformerLayer(hidden_dim, hidden_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
            for _ in range(n_layers - 1)
        ])
        self.layers.append(
            GraphTransformerLayer(hidden_dim, out_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
        )
    
    def forward(self, data):
        data = data.to(self.device)
        h = data.x.float().to(self.device)
        h_lap_pos_enc = data.lap_pos_enc.to(self.device)
        e = data.edge_attr.float().to(self.device)
        edge_index = data.edge_index.to(self.device)
        
        h = self.linear_h(h)
        h_lap_pos_enc = self.embedding_lap_pos_enc(h_lap_pos_enc.float())
        h = h + h_lap_pos_enc
        h = self.in_feat_dropout(h)
        
        e = self.linear_e(e)
        
        for conv in self.layers:
            h, e = conv(h, edge_index, e)
        
        return h

In [11]:
class SelfAttention(nn.Module):
    def __init__(self, hid_dim, n_heads, dropout, device):
        super().__init__()
        
        self.hid_dim = hid_dim
        self.n_heads = n_heads
        assert hid_dim % n_heads == 0
        
        self.w_q = nn.Linear(hid_dim, hid_dim)
        self.w_k = nn.Linear(hid_dim, hid_dim)
        self.w_v = nn.Linear(hid_dim, hid_dim)
        self.fc = nn.Linear(hid_dim, hid_dim)
        self.do = nn.Dropout(dropout)
        self.scale = torch.sqrt(torch.FloatTensor([hid_dim // n_heads])).to(device)
    
    def forward(self, query, key, value, mask=None):
        bsz = query.shape[0]
        
        Q = self.w_q(query)
        K = self.w_k(key)
        V = self.w_v(value)
        
        Q = Q.view(bsz, -1, self.n_heads, self.hid_dim // self.n_heads).permute(0, 2, 1, 3)
        K = K.view(bsz, -1, self.n_heads, self.hid_dim // self.n_heads).permute(0, 2, 1, 3)
        V = V.view(bsz, -1, self.n_heads, self.hid_dim // self.n_heads).permute(0, 2, 1, 3)
        
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale
        
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e10)
        
        attention = self.do(F.softmax(energy, dim=-1))
        x = torch.matmul(attention, V)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, -1, self.n_heads * (self.hid_dim // self.n_heads))
        x = self.fc(x)
        
        return x

In [12]:
def pyg_batch_split(bg, feats):
    """
    Splits PyG batch into individual graphs, padding to max size.
    Exact DGL logic, just swapping data accessors
    """
    # Only difference: PyG needs to compute batch_num_nodes
    batch = bg.batch
    batch_num_nodes = scatter(
        torch.ones_like(batch), batch, 
        dim=0, dim_size=bg.num_graphs, reduce='sum'
    )
    
    # Now follow DGL exactly
    max_num_nodes = int(batch_num_nodes.max())
    cum_nodes = torch.cat([batch.new_zeros(1), batch_num_nodes.cumsum(dim=0)])
    idx = torch.arange(feats.size(0), dtype=torch.long, device=feats.device)
    idx = (idx - cum_nodes[batch]) + (batch * max_num_nodes)
    size = [bg.num_graphs * max_num_nodes] + list(feats.size())[1:]
    out = feats.new_full(size, fill_value=0)
    out[idx] = feats
    out = out.view([bg.num_graphs, max_num_nodes] + list(feats.size())[1:])
    return out

In [ ]:
class MDGTDTInet(nn.Module):
    def __init__(self, compound_dim=128, protein_dim=128, gt_layers=10, gt_heads=8, out_dim=1,
                 compound_node_dim=44, compound_edge_dim=10, protein_node_dim=41, protein_edge_dim=5,
                 device=None):
        super(MDGTDTInet, self).__init__()
        
        # Set device - use provided device or default to cuda if available
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.device = device
        
        self.compound_dim = compound_dim
        self.protein_dim = protein_dim
        self.n_layers = gt_layers
        self.n_heads = gt_heads
        
        self.crossAttention = SelfAttention(hid_dim=self.compound_dim, n_heads=1, dropout=0.2, device=self.device)
        
        self.compound_gt = GraphTransformerCompound(
            self.device, n_layers=gt_layers, node_dim=44, edge_dim=compound_edge_dim, 
            hidden_dim=compound_dim, out_dim=compound_dim, n_heads=gt_heads, 
            in_feat_dropout=0.0, dropout=0.2, pos_enc_dim=8
        )
        
        self.protein_gt = GraphTransformerProtein(
            self.device, n_layers=gt_layers, node_dim=1321, edge_dim=protein_edge_dim, 
            hidden_dim=protein_dim, out_dim=protein_dim, n_heads=gt_heads, 
            in_feat_dropout=0.0, dropout=0.2, pos_enc_dim=8
        )
        
        self.protein_embedding_fc = nn.Linear(320, self.protein_dim)
        
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()
        self.joint_attn_prot = nn.Linear(compound_dim, compound_dim)
        self.joint_attn_comp = nn.Linear(compound_dim, compound_dim)
        
        self.classifier = nn.Sequential(
            nn.Linear(compound_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, out_dim)
        )
    
    def forward(self, compound_graph, protein_graph, protein_embedding):
        # Compound encoding
        compound_feat = self.compound_gt(compound_graph)
        compound_feat_x = pyg_batch_split(compound_graph, compound_feat)
        compound_feats = compound_feat_x
        
        # Protein encoding
        protein_feat = self.protein_gt(protein_graph)
        protein_feat_x = pyg_batch_split(protein_graph, protein_feat)
        protein_embedding = self.protein_embedding_fc(protein_embedding)
        protein_feats = self.crossAttention(protein_embedding, protein_feat_x, protein_feat_x)
        
        # Compound-protein interaction
        inter_comp_prot = self.sigmoid(
            torch.einsum('bij,bkj->bik', 
                        self.joint_attn_prot(self.relu(protein_feats)), 
                        self.joint_attn_comp(self.relu(compound_feats)))
        )
        inter_comp_prot_sum = torch.einsum('bij->b', inter_comp_prot)
        inter_comp_prot = torch.einsum('bij,b->bij', inter_comp_prot, 1/inter_comp_prot_sum)
        
        # Compound-protein joint embedding
        cp_embedding = self.tanh(torch.einsum('bij,bkj->bikj', protein_feats, compound_feats))
        cp_embedding = torch.einsum('bijk,bij->bk', cp_embedding, inter_comp_prot)
        
        x = self.classifier(cp_embedding)
        
        return x

## Section 3: Evaluation Metrics

In [14]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    return np.sqrt(mse(y_true, y_pred))

def ci(y_true, y_pred):
    """Concordance Index"""
    ind = np.argsort(y_true)
    y_true = y_true[ind]
    y_pred = y_pred[ind]
    i = len(y_true) - 1
    j = i - 1
    z = 0.0
    S = 0.0
    while i > 0:
        while j >= 0:
            if y_true[i] > y_true[j]:
                z = z + 1
                u = y_pred[i] - y_pred[j]
                if u > 0:
                    S = S + 1
                elif u == 0:
                    S = S + 0.5
            j = j - 1
        i = i - 1
        j = i - 1
    ci = S / z
    return ci

def rm2(y_true, y_pred):
    """R squared metric"""
    r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2)
    r02 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true ** 2))
    return r2 if r2 > r02 else r02

## Section 4: Training and Testing Functions

In [15]:
def train(model, device, train_loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for batch_idx, data in enumerate(train_loader):
        compound_graph, protein_graph, protein_embedding, label = data
        
        compound_graph = compound_graph.to(device)
        protein_graph = protein_graph.to(device)
        protein_embedding = protein_embedding.to(device)
        label = label.to(device)
        
        output = model(compound_graph, protein_graph, protein_embedding)
        loss = criterion(output, label.view(-1, 1).float())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        if (batch_idx + 1) % 10 == 0:
            print(f'  Batch {batch_idx + 1}/{len(train_loader)}, Loss: {loss.item():.6f}')
    
    avg_loss = total_loss / num_batches
    return avg_loss

def test(model, device, test_loader):
    model.eval()
    total_preds = torch.Tensor()
    total_labels = torch.Tensor()
    
    print(f"  🔍 Starting test phase with {len(test_loader)} batches...")
    
    with torch.no_grad():
        for batch_idx, data in enumerate(test_loader):
            try:
                compound_graph, protein_graph, protein_embedding, label = data
                
                if batch_idx == 0:
                    print(f"  📊 Batch 0 shapes:")
                    print(f"    - protein_embedding shape: {protein_embedding.shape}")
                    print(f"    - label shape: {label.shape}")
                    print(f"    - compound_graph.num_nodes: {compound_graph.num_nodes}")
                    print(f"    - protein_graph.num_nodes: {protein_graph.num_nodes}")
                
                compound_graph = compound_graph.to(device)
                protein_graph = protein_graph.to(device)
                protein_embedding = protein_embedding.to(device)
                
                output = model(compound_graph, protein_graph, protein_embedding)
                
                if batch_idx == 0:
                    print(f"    - output shape: {output.shape}")
                    print(f"    - output values (first 3): {output[:3]}")
                
                total_preds = torch.cat((total_preds, output.cpu()), 0)
                total_labels = torch.cat((total_labels, label.view(-1, 1).cpu()), 0)
                
                if (batch_idx + 1) % 50 == 0:
                    print(f"  ✓ Processed {batch_idx + 1}/{len(test_loader)} test batches")
                    
            except Exception as e:
                print(f"  ❌ Error in batch {batch_idx}: {str(e)}")
                import traceback
                traceback.print_exc()
                continue
    
    print(f"  📈 Total predictions collected: {len(total_preds)}")
    print(f"  📈 Total labels collected: {len(total_labels)}")
    
    if len(total_labels) == 0:
        print("  ❌ No predictions collected!")
        return float('nan'), float('nan'), float('nan'), float('nan')
    
    total_labels = total_labels.numpy().flatten()
    total_preds = total_preds.numpy().flatten()
    
    print(f"  📊 Final shapes - preds: {total_preds.shape}, labels: {total_labels.shape}")
    print(f"  📊 Pred range: [{total_preds.min():.4f}, {total_preds.max():.4f}]")
    print(f"  📊 Label range: [{total_labels.min():.4f}, {total_labels.max():.4f}]")
    
    try:
        MSE = mse(total_labels, total_preds)
        RMSE = rmse(total_labels, total_preds)
        CI = ci(total_labels, total_preds)
        RM2 = rm2(total_labels, total_preds)
        
        print(f"  ✓ Metrics computed successfully")
        print(f"    - MSE: {MSE:.6f}")
        print(f"    - RMSE: {RMSE:.6f}")
        print(f"    - CI: {CI:.5f}")
        print(f"    - RM2: {RM2:.5f}")
        
    except Exception as e:
        print(f"  ❌ Error computing metrics: {str(e)}")
        import traceback
        traceback.print_exc()
        return float('nan'), float('nan'), float('nan'), float('nan')
    
    return MSE, RMSE, CI, RM2

## Section 5: Main Training Pipeline

Load real data and train the model

In [68]:
# ============================================================================
# LOAD REAL DATA
# ============================================================================
print("="*80)
print("LOADING DAVIS DATASET - FOLD 1")
print("="*80)

# Load data
train_loader, test_loader, train_dataset, test_dataset = load_real_dataset(
    dataset='Davis',
    fold=1,
    batch_size=4,
    num_workers=0,
    compound_node_dim=compound_node_dim,
    compound_edge_dim=compound_edge_dim,
    protein_node_dim=protein_node_dim,
    protein_edge_dim=protein_edge_dim
)

print(f"\n✓ Data loaders created successfully")
print(f"  - Train loader: {len(train_loader)} batches")
print(f"  - Test loader: {len(test_loader)} batches")

LOADING DAVIS DATASET - FOLD 1
Loading CSV from: AttentionMGT/Data/Davis/DTA/fold/1/Davis_train.csv
✓ Loaded train split: 19964 samples
  - Data directory: AttentionMGT/
  - Compound graphs dir: AttentionMGT/data/Davis/Processed/compound_graph/
  - Compound graphs available: True
  - Protein graphs available: True
  - ESM embeddings available: True
Loading CSV from: AttentionMGT/Data/Davis/DTA/fold/1/Davis_test.csv
✓ Loaded test split: 4992 samples
  - Data directory: AttentionMGT/
  - Compound graphs dir: AttentionMGT/data/Davis/Processed/compound_graph/
  - Compound graphs available: True
  - Protein graphs available: True
  - ESM embeddings available: True

✓ Data loaders created successfully
  - Train loader: 4991 batches
  - Test loader: 1248 batches


In [ ]:
# # ============================================================================
# # INITIALIZE MODEL
# # ============================================================================
# print("\n" + "="*80)
# print("INITIALIZING MODEL")
# print("="*80)

# model = MDGTDTInet(
#     compound_dim=128,
#     protein_dim=128,
#     gt_layers=10,
#     gt_heads=8,
#     out_dim=1
# )

# model.to(device)

# # Count parameters
# total_params = sum(p.numel() for p in model.parameters())
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# print(f"\n✓ Model initialized on device: {device}")
# print(f"  - Total parameters: {total_params:,}")
# print(f"  - Trainable parameters: {trainable_params:,}")


INITIALIZING MODEL

✓ Model initialized on device: cpu
  - Total parameters: 6,228,609
  - Trainable parameters: 6,228,609


In [ ]:
# # ============================================================================
# # TRAINING SETUP
# # ============================================================================
# print("\n" + "="*80)
# print("TRAINING SETUP")
# print("="*80)

# # Hyperparameters
# lr = 1e-4
# epochs = 50
# patience = 10

# # Optimizer
# optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# # Learning rate scheduler
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer=optimizer, 
#     mode='min', 
#     factor=0.8, 
#     patience=patience, 
#     min_lr=1e-5
# )

# # Loss function
# criterion = nn.MSELoss()

# print(f"\n✓ Training configuration:")
# print(f"  - Learning rate: {lr}")
# print(f"  - Epochs: {epochs}")
# print(f"  - Optimizer: Adam")
# print(f"  - Loss function: MSELoss")
# print(f"  - Batch size: 4")
# print(f"  - Scheduler: ReduceLROnPlateau (patience={patience})")


TRAINING SETUP

✓ Training configuration:
  - Learning rate: 0.0001
  - Epochs: 50
  - Optimizer: Adam
  - Loss function: MSELoss
  - Batch size: 4
  - Scheduler: ReduceLROnPlateau (patience=10)


In [ ]:
# # ============================================================================
# # TRAINING LOOP
# # ============================================================================
# print("\n" + "="*80)
# print("TRAINING")
# print("="*80)

# best_mse = float('inf')
# best_epoch = -1
# no_improve_count = 0
# training_history = []

# start_time = timeit.default_timer()

# print(f"\n{'Epoch':<8} {'Time(s)':<12} {'Train Loss':<15} {'Test MSE':<15} {'RMSE':<15} {'CI':<12} {'RM2':<12}")
# print("-" * 100)

# for epoch in range(epochs):
#     epoch_start = timeit.default_timer()
    
#     # Train
#     train_loss = train(model, device, train_loader, optimizer, criterion)
    
#     # Test
#     mse_test, rmse_test, ci_test, rm2_test = test(model, device, test_loader)
    
#     # Step scheduler
#     scheduler.step(mse_test)
    
#     epoch_end = timeit.default_timer()
#     epoch_time = epoch_end - epoch_start
    
#     # Store history
#     training_history.append({
#         'epoch': epoch + 1,
#         'train_loss': train_loss,
#         'test_mse': mse_test,
#         'test_rmse': rmse_test,
#         'test_ci': ci_test,
#         'test_rm2': rm2_test,
#         'time': epoch_time
#     })
    
#     # Print
#     print(f"{epoch + 1:<8} {epoch_time:<12.2f} {train_loss:<15.6f} {mse_test:<15.6f} {rmse_test:<15.6f} {ci_test:<12.5f} {rm2_test:<12.5f}")
    
#     # Check improvement
#     if mse_test < best_mse:
#         best_mse = mse_test
#         best_epoch = epoch + 1
#         no_improve_count = 0
#         print(f"        ✓ MSE improved to {best_mse:.6f}")
        
#         # Save best model
#         torch.save(model.state_dict(), 'best_model.pt')
#     else:
#         no_improve_count += 1
#         if no_improve_count >= patience:
#             print(f"\n✗ Early stopping at epoch {epoch + 1} (no improvement for {patience} epochs)")
#             break

# end_time = timeit.default_timer()
# total_time = end_time - start_time

# print("-" * 100)
# print(f"\n✅ Training completed!")
# print(f"  - Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
# print(f"  - Best epoch: {best_epoch}")
# print(f"  - Best MSE: {best_mse:.6f}")

## Section 6: Results Summary

In [ ]:
# # Create results DataFrame
# results_df = pd.DataFrame(training_history)

# print("\n" + "="*80)
# print("TRAINING RESULTS SUMMARY")
# print("="*80)

# print("\n📊 Final Model Performance:")
# best_result = results_df[results_df['epoch'] == best_epoch].iloc[0]
# print(f"  - Epoch: {best_result['epoch']}")
# print(f"  - Train Loss: {best_result['train_loss']:.6f}")
# print(f"  - Test MSE: {best_result['test_mse']:.6f}")
# print(f"  - Test RMSE: {best_result['test_rmse']:.6f}")
# print(f"  - Concordance Index (CI): {best_result['test_ci']:.5f}")
# print(f"  - RM² Score: {best_result['test_rm2']:.5f}")

# print("\n📈 Overall Statistics:")
# print(f"  - Average epoch time: {results_df['time'].mean():.2f} seconds")
# print(f"  - Min test MSE: {results_df['test_mse'].min():.6f}")
# print(f"  - Max test MSE: {results_df['test_mse'].max():.6f}")
# print(f"  - Mean test MSE: {results_df['test_mse'].mean():.6f}")
# print(f"  - Max CI: {results_df['test_ci'].max():.5f}")
# print(f"  - Max RM²: {results_df['test_rm2'].max():.5f}")

# # Save results
# results_df.to_csv('training_results.csv', index=False)
# print(f"\n✓ Results saved to: training_results.csv")

In [ ]:
# import matplotlib.pyplot as plt

# # Plot training results
# fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# # Plot 1: MSE over epochs
# axes[0, 0].plot(results_df['epoch'], results_df['test_mse'], 'b-o', linewidth=2, markersize=6)
# axes[0, 0].axhline(y=best_mse, color='g', linestyle='--', linewidth=2, label=f'Best MSE: {best_mse:.6f}')
# axes[0, 0].axvline(x=best_epoch, color='r', linestyle='--', linewidth=2, alpha=0.7)
# axes[0, 0].set_xlabel('Epoch', fontsize=12)
# axes[0, 0].set_ylabel('MSE', fontsize=12)
# axes[0, 0].set_title('Test MSE over Epochs', fontsize=14, fontweight='bold')
# axes[0, 0].legend(fontsize=10)
# axes[0, 0].grid(alpha=0.3)

# # Plot 2: RMSE over epochs
# axes[0, 1].plot(results_df['epoch'], results_df['test_rmse'], 'g-s', linewidth=2, markersize=6)
# axes[0, 1].set_xlabel('Epoch', fontsize=12)
# axes[0, 1].set_ylabel('RMSE', fontsize=12)
# axes[0, 1].set_title('Test RMSE over Epochs', fontsize=14, fontweight='bold')
# axes[0, 1].grid(alpha=0.3)

# # Plot 3: CI over epochs
# axes[1, 0].plot(results_df['epoch'], results_df['test_ci'], 'r-^', linewidth=2, markersize=6)
# axes[1, 0].set_xlabel('Epoch', fontsize=12)
# axes[1, 0].set_ylabel('Concordance Index', fontsize=12)
# axes[1, 0].set_title('Test CI over Epochs', fontsize=14, fontweight='bold')
# axes[1, 0].grid(alpha=0.3)

# # Plot 4: RM² over epochs
# axes[1, 1].plot(results_df['epoch'], results_df['test_rm2'], 'purple', marker='d', linewidth=2, markersize=6)
# axes[1, 1].set_xlabel('Epoch', fontsize=12)
# axes[1, 1].set_ylabel('RM² Score', fontsize=12)
# axes[1, 1].set_title('Test RM² over Epochs', fontsize=14, fontweight='bold')
# axes[1, 1].grid(alpha=0.3)

# plt.tight_layout()
# plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
# plt.show()

# print("\n✓ Plots saved to: training_results.png")